# Eternal Dao — Episode 5: free Colab production
**Production-only notebook.** Reads private Docs/Slides through your Google sign-in; the public repository does not contain any manuscript or images. Original Kokoro `am_michael` voice at 1.25×, music-first, 50 ordered visuals, 1080p 24fps. The rendered file goes into your existing private Episode 5 Google Drive folder as **PRIVATE_REVIEW**, never as automatically approved/public content.

**How to use:** In Colab, choose **Runtime → Change runtime type → T4 GPU** if offered, then **Runtime → Run all** and authorize Google Drive access. If no GPU is offered the CPU can run but will be slower. Free Colab resources are not guaranteed. This notebook does not fabricate any of the user-required human final-video review scores.


In [ ]:
# ETERNAL DAO — EPISODE 05 — PRIVATE FREE GOOGLE COLAB BUILDER
# The only public item is this production code. Manuscript and images are
# fetched from the owner's private Google Drive with their own Google login.
import os, sys, subprocess, json, re, hashlib, time, pathlib, math, shutil
from pathlib import Path

def run(cmd, quiet=False):
    r = subprocess.run(cmd, text=True,
                       stdout=subprocess.PIPE if quiet else None,
                       stderr=subprocess.PIPE if quiet else None)
    if r.returncode:
        raise RuntimeError("Command failed: "+str(cmd)+"\n"+(r.stderr or ""))
    return r

print("Setting up original Kokoro-82M, FFmpeg and Google Drive access.")
run(["apt-get","update","-qq"], quiet=True)
run(["apt-get","install","-qq","-y","espeak-ng","ffmpeg"], quiet=True)
run([sys.executable,"-m","pip","install","-q",
     "kokoro>=0.9.4","soundfile","google-api-python-client",
     "google-auth-httplib2","google-auth-oauthlib","pillow","requests"])
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from PIL import Image, ImageOps
import requests
import numpy as np
import soundfile as sf
import torch
from kokoro import KPipeline

SCOPES = ["https://www.googleapis.com/auth/drive","https://www.googleapis.com/auth/presentations.readonly"]
credentials, _ = google.auth.default(scopes=SCOPES)
drive = build("drive","v3",credentials=credentials,cache_discovery=False)
slides_api = build("slides","v1",credentials=credentials,cache_discovery=False)
DOC_ID = "1k6VXjraGy13D2uKonZx6OQX2NENuOhBZlP8-O64wPhY"
SLIDES_ID = "1euBbTVJmWzrnijAcNKSVljuzBpmED-yqAz4gc9mfZ0E"
OUTPUT_FOLDER_ID = "1oVFMbPIdpaEvuEKIGB2UG7ojZBG78M8S"
W = Path("/content/eternal_dao_ep05_r2")
W.mkdir(parents=True,exist_ok=True)
(W/"art").mkdir(exist_ok=True)
(W/"voice").mkdir(exist_ok=True)
(W/"segments").mkdir(exist_ok=True)
source = drive.files().export(fileId=DOC_ID,mimeType="text/plain").execute().decode("utf-8-sig")
source=source.replace("\r\n","\n").strip()
assert all(x in source for x in ("CHAPTER NINE","CHAPTER TEN","CHAPTER ELEVEN")), "Wrong script"
assert source.count("CHAPTER NINE")==1 and source.count("CHAPTER TEN")==1 and source.count("CHAPTER ELEVEN")==1
assert len(source.split())>=5000, "Incomplete narration source"
(W/"r2_script.txt").write_text(source,encoding="utf-8")
script_sha=hashlib.sha256(source.encode()).hexdigest()
presentation=slides_api.presentations().get(presentationId=SLIDES_ID).execute()
ordered=presentation.get("slides",[])
assert len(ordered)==50, "Expected the 50 ordered illustration slides"
image_meta=[]
for idx,page in enumerate(ordered,1):
    sid="edao05_s"+str(idx).zfill(2)
    assert page["objectId"]==sid, "Out-of-order slide "+str(idx)
    images=[el for el in page.get("pageElements",[]) if "image" in el]
    assert len(images)==1, "Missing/duplicate image on "+sid
    image=images[0]["image"]
    srcurl=image.get("contentUrl") or image.get("sourceUrl")
    assert srcurl and srcurl.startswith("https://"), "Missing source for "+sid
    image_meta.append({"index":idx,"slide_id":sid,"url":srcurl})
assert len(set(x["url"] for x in image_meta))==50,"Duplicate image asset sources"
for item in image_meta:
    path=W/"art"/("scene_"+str(item["index"]).zfill(2)+".jpg")
    if path.exists():
        continue
    resp=requests.get(item["url"],timeout=90)
    resp.raise_for_status()
    from io import BytesIO
    raw=Image.open(BytesIO(resp.content)).convert("RGB")
    assert min(raw.size)>250,"Bad source resolution"
    # No crop or stretched faces. Image is reframed with edge-blur only if
    # its aspect ratio differs. A reviewer must inspect the final result.
    fg=ImageOps.contain(raw,(1920,1080),method=Image.Resampling.LANCZOS)
    bg=ImageOps.fit(raw,(1920,1080),method=Image.Resampling.LANCZOS)
    if fg.size!=(1920,1080):
        from PIL import ImageFilter
        bg=bg.filter(ImageFilter.GaussianBlur(30))
        bg.paste(fg,((1920-fg.width)//2,(1080-fg.height)//2))
        out=bg
    else:
        out=fg
    out.save(path,"JPEG",quality=94,subsampling=0)
    item["image_sha256"]=hashlib.sha256(path.read_bytes()).hexdigest()
print("Loaded PRIVATE approved draft, %s words; 50 ordered original illustrations." % len(source.split()))
print("Source SHA256",script_sha)
print("Free platform: Google Colab. GPU available:",torch.cuda.is_available())


In [ ]:
# MUSIC-FIRST RULE: create and hash original score BEFORE the first TTS call.
# Quiet original atmospheric drone, gentle harmony, 100% generated here.
import numpy as np
SR=24000
secs=64
t=np.arange(secs*SR,dtype=np.float32)/SR
scale=(130.81,196.0,261.63,392.0)
slow=np.zeros_like(t)
for j,frequency in enumerate(scale):
    drift=0.008*np.sin(2*np.pi*(0.047+j*.008)*t+j)
    slow += (0.19/(j+1))*np.sin(2*np.pi*frequency*t + drift)
    slow += (0.09/(j+1))*np.sin(2*np.pi*(frequency/2)*t + 0.23)
slow*=((1-np.exp(-t/.95))*np.minimum(1,(secs-t)/1.6))
slow=np.clip(slow,-0.8,0.8)
musicpath=W/"original_music_FIRST.wav"
sf.write(str(musicpath),slow,SR,subtype="PCM_16")
music_receipt={"music_sha256":hashlib.sha256(musicpath.read_bytes()).hexdigest(),
 "generated_at_utc":__import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat(),
 "source_sha256":script_sha,"music_before_voice":True,"voice":"am_michael","speed":1.25}
(W/"music_first_receipt.json").write_text(json.dumps(music_receipt,indent=2),encoding="utf-8")
print("Original music-first receipt:",music_receipt["music_sha256"])


In [ ]:
# Prepare exactly 50 consecutive text scenes without omitting narration.
# This is a provisional scene split. The image/event matching review remains
# compulsory even when all source and technical checks pass.
units=re.split(r'(?<=[.!?])\s+(?=(?:["“]?[A-Z0-9]))',source)
units=[x.strip() for x in units if x.strip()]
assert len(units)>50, "Cannot divide script reliably"
scene_parts=[]
cursor=0
total=sum(len(x.split()) for x in units)
for i in range(50):
    remaining=50-i
    if remaining==1:
        stop=len(units)
    else:
        budget=round((total-sum(len(x.split()) for x in units[:cursor]))/remaining)
        stop=cursor
        words=0
        while stop<len(units)-(remaining-1) and (words<budget or stop==cursor):
            words+=len(units[stop].split())
            stop+=1
    piece=" ".join(units[cursor:stop]).strip()
    assert piece
    scene_parts.append(piece)
    cursor=stop
assert cursor==len(units)
def compact(s):return re.sub(r'\s+',' ',s).strip()
assert compact(" ".join(scene_parts))==compact(source),"Source reconstruction failure"
# Keep approvals strictly private; no staged draft can claim signed scores.
(W/"scene_source_50.json").write_text(json.dumps([
    {"scene":i+1,"slide_id":image_meta[i]["slide_id"],
     "spoken_text":text,"spoken_sha256":hashlib.sha256(text.encode()).hexdigest(),
     "image_sha256":image_meta[i]["image_sha256"],
     "voice_image_score":"UNREVIEWED"} for i,text in enumerate(scene_parts)],ensure_ascii=False,indent=2))
print("Exactly 50 complete narration chunks source-matched.")
# Reuse complete valid scene WAVs when a free Colab runtime disconnects,
# if the user has saved the working folder. No reuse from OLD D03 narration.
pipeline=KPipeline(lang_code="a",device="cuda" if torch.cuda.is_available() else "cpu")
timing=[]
for i,text in enumerate(scene_parts,1):
    path=W/"voice"/("scene_"+str(i).zfill(2)+".wav")
    if not path.exists():
        blocks=[]
        for graphemes,phonemes,audio in pipeline(text,voice="am_michael",speed=1.25):
            if audio is None:
                raise RuntimeError("No Kokoro audio for scene "+str(i))
            arr=audio.cpu().numpy() if hasattr(audio,"cpu") else np.asarray(audio)
            blocks.append(arr.astype(np.float32).reshape(-1))
            blocks.append(np.zeros(int(0.12*SR),dtype=np.float32))
        if not blocks:
            raise RuntimeError("Kokoro returned no audio for scene "+str(i))
        wave=np.concatenate(blocks)
        sf.write(str(path),wave,SR,subtype="PCM_16")
    info=sf.info(str(path))
    assert info.samplerate==SR and info.duration>1, "Bad audio: "+str(i)
    timing.append({"scene":i,"seconds":info.duration,
                   "spoken_sha256":hashlib.sha256(text.encode()).hexdigest(),
                   "wav_sha256":hashlib.sha256(path.read_bytes()).hexdigest()})
    if i%10==0:print("Voice generated:",i,"/ 50 scenes")
# Concatenate the exact new R2 narration. No old MP4 or old D03 voice reused.
from scipy.io import wavfile
voice_parts=[]
cumulative=0.0
for tinfo in timing:
    rate,wav=wavfile.read(str(W/"voice"/("scene_"+str(tinfo["scene"]).zfill(2)+".wav")))
    assert rate==SR
    voice_parts.append(wav)
    tinfo["start_seconds"]=round(cumulative,4)
    cumulative+=len(wav)/SR
    tinfo["end_seconds"]=round(cumulative,4)
voice=np.concatenate(voice_parts)
narration=W/"R2_Kokoro_Michael_Narration.wav"
wavfile.write(str(narration),SR,voice)
(W/"scene_timing.json").write_text(json.dumps(timing,indent=2))
assert narration.exists() and narration.stat().st_size>5_000_000
assert musicpath.stat().st_mtime<=narration.stat().st_mtime,"Music-first sequence invalid"
print("New R2 narration duration:",round(cumulative/60,2),"minutes")


In [ ]:
# Strict private assembly. This generates a candidate, not publication approval.
# 50 per-scene slow push-ins are timed to each NEW voice clip.
for info in timing:
    idx=info["scene"]
    still=W/"art"/("scene_"+str(idx).zfill(2)+".jpg")
    out=W/"segments"/("scene_"+str(idx).zfill(2)+".mp4")
    frames=max(1,round(info["seconds"]*24))
    if out.exists():
        continue
    # Real geometric motion at source-matched 24fps (no text on images).
    z="min(zoom+0.00007,1.07)"
    xf="(iw-iw/zoom)*(on/%d)" % max(1,frames)
    yf="(ih-ih/zoom)*0.45"
    vf=("zoompan=z='%s':x='%s':y='%s':d=%s:"
        "s=1920x1080:fps=24,format=yuv420p" % (z,xf,yf,frames))
    run(["ffmpeg","-hide_banner","-loglevel","error","-y",
         "-i",str(still),"-vf",vf,"-frames:v",str(frames),
         "-r","24","-c:v","libx264","-preset","veryfast",
         "-crf","21","-pix_fmt","yuv420p",
         "-movflags","+faststart",str(out)])
    if idx%10==0:print("Rendered scene",idx,"/ 50")
concat=W/"scene_concat.ffconcat"
concat.write_text("ffconcat version 1.0\n"+"".join(
    "file '"+str(W/"segments"/("scene_"+str(i).zfill(2)+".mp4"))+"'\n"
    for i in range(1,51)))
out=W/"Eternal_Dao_E05_R2_PRIVATE_REVIEW_1080p.mp4"
run(["ffmpeg","-hide_banner","-loglevel","error","-y",
     "-safe","0","-f","concat","-i",str(concat),
     "-i",str(narration),"-stream_loop","-1","-i",str(musicpath),
     "-filter_complex",
     "[1:a]volume=1.0[v];[2:a]lowpass=f=600,volume=0.065[bg];"
     "[v][bg]amix=inputs=2:duration=first:dropout_transition=0,"
     "loudnorm=I=-17:TP=-1.5:LRA=9[a]",
     "-map","0:v:0","-map","[a]","-t",str(round(cumulative,3)),
     "-c:v","copy","-c:a","aac","-b:a","192k",
     "-movflags","+faststart",str(out)])
probe=json.loads(subprocess.check_output([
 "ffprobe","-v","error","-show_entries","format=duration,size:"
 "stream=codec_name,codec_type,width,height,r_frame_rate",
 "-of","json",str(out)],text=True))
d=float(probe["format"]["duration"])
assert abs(d-cumulative)<1.2,(d,cumulative)
assert any(x.get("codec_name")=="h264" and x.get("width")==1920 for x in probe["streams"])
assert any(x.get("codec_name")=="aac" for x in probe["streams"])
# Actually decode the completed private master to catch broken streams.
run(["ffmpeg","-v","error","-i",str(out),"-f","null","-"],quiet=True)
final_sha=hashlib.sha256(out.read_bytes()).hexdigest()
receipt={"status":"PRIVATE_TECHNICAL_CANDIDATE_NOT_RELEASE_APPROVED",
 "novel":"Eternal Dao of the Lost Epoch","episode":5,
 "source_sha256":script_sha,"final_video_sha256":final_sha,
 "final_duration_seconds":d,"scenes":len(timing),"video_streams":probe["streams"],
 "music_first":music_receipt,
 "human_audio_review":"NOT_COMPLETED",
 "human_image_to_voice_review":"NOT_COMPLETED",
 "all_four_scene_scores_8":"NOT_COMPLETED",
 "story_author_approval":"NOT_VERIFIED",
 "publication":"BLOCKED"}
qa=W/"R2_PRIVATE_BUILD_QA.json"
qa.write_text(json.dumps(receipt,indent=2))
print("New private master rendered:",out.name,"Size MiB:",round(out.stat().st_size/(1024*1024),1))
print("Source voice and H.264/AAC decoding checked; image/voice artistic approval still required.")
# Upload ONLY to the private authorized Episode 5 Google Drive folder.
def upload_file(path):
    meta={"name":path.name,"parents":[OUTPUT_FOLDER_ID],
          "description":"PRIVATE QUALITY REVIEW ONLY — do not upload to YouTube"}
    media=MediaFileUpload(str(path),mimetype="video/mp4" if path.suffix==".mp4" else "application/json",
                          chunksize=8*1024*1024,resumable=True)
    req=drive.files().create(body=meta,media_body=media,fields="id,name,webViewLink")
    result=None
    while result is None:
        _,result=req.next_chunk(num_retries=3)
    return result
video_drive=upload_file(out)
qa_drive=upload_file(qa)
print("SAVED TO PRIVATE DRIVE FOLDER:",OUTPUT_FOLDER_ID)
print("VIDEO PRIVATE QA FILE ID:",video_drive["id"])
print("QA RECEIPT:",qa_drive["id"])
print("Do not publish; independent sound, visual, source and owner release checks remain.")
